
# Chapter 5 Full Python Tutorial

This notebook is a Python-first walkthrough of `ch.5. Deep Learning Algorithms`.

Goal of this chapter:
- understand what data representation each model family expects
- understand why chapter 5 does **not** consume the chapter 3/4 sample `.mat` files directly
- run the shipped PyTorch models with dummy tensors so the tensor flow is concrete
- inspect the gap between the reference research code and a portable local learning setup



## Source coverage

This notebook covers the main reference code under chapter 5:

- `PyTorch/CVNN/AlexNet.py`
- `PyTorch/CVNN/Resnet.py`
- `PyTorch/CVNN/Transformer.py`
- `PyTorch/CVNN/train_dataset.py`
- `PyTorch/RVNN/ResNet.py`
- `PyTorch/RVNN/train_dataset.py`
- `PyTorch/SEN/SEN_trainer.py`
- `PyTorch/SEN/SEN_TEST.py`

Important scope note:
- the repository ships **model code** and `shuffle.txt` path logs
- it does **not** ship the full Widar / Widar3 training dataset
- so this notebook focuses on architecture, tensor semantics, preprocessing assumptions, and forward-pass behavior rather than end-to-end training accuracy



## Representation mismatch between chapter 3/4 and chapter 5

The most important thing to internalize before reading the deep-learning code is this:

- chapter 3 and chapter 4 sample files are **tutorial CSI tensors** with shape `[T, S, A, L]`
- chapter 5 models expect **task-specific learned representations** that already look different

In other words, chapter 5 is not just “take `csi_src_test.mat` and call `train.py`”.
There is a representation shift between the earlier signal-processing chapters and the deep-learning baselines.


In [1]:
from __future__ import annotations

from pathlib import Path
import os
import numpy as np
from pprint import pprint
import importlib.util

from scipy.io import loadmat

CACHE_DIR = Path.cwd() / ".cache"
CACHE_DIR.mkdir(exist_ok=True)

os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_DIR))

MPL_CACHE_DIR = Path.cwd() / ".matplotlib"
MPL_CACHE_DIR.mkdir(exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))


'/Users/doanvankhoan/Desktop/wifi-sensing-learning/chapter5/notebooks/.matplotlib'

In [2]:
try:
    import matplotlib
    try:
        from IPython import get_ipython
    except Exception:
        get_ipython = None

    ip = get_ipython() if get_ipython is not None else None

    if ip is None:
        matplotlib.use("Agg")
    else:
        try:
            ip.run_line_magic("matplotlib", "inline")
        except:
            pass

    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True

except Exception:
    MATPLOTLIB_AVAILABLE = False

In [3]:
import torch
from einops import rearrange

In [4]:
ch3_csi = loadmat("../../chapter3/data/csi_src_test.mat")['csi']
ch4_csi = loadmat("../../chapter4/data/csi_src_test.mat")['csi']

In [ ]:
with open("../Pytorch/RVNN/shuffle.txt", 'r') as f:
    rvnn_paths = [line.strip() for line in f if line.strip()]
with open("../Pytorch/CVNN/shuffle.txt", 'r') as f:
    cvnn_paths = [line.strip() for line in f if line.strip()]

In [41]:
print('chapter 3 sample csi shape:', ch3_csi.shape)
print('chapter 4 sample csi shape:', ch4_csi.shape)
print('first RVNN path:', rvnn_paths[0])
print('first CVNN path:', cvnn_paths[0])

chapter 3 sample csi shape: (999, 57, 3, 2)
chapter 4 sample csi shape: (999, 57, 3, 2)
first RVNN path: /srv/dataset/widar_dfs/DFS/20181109/user1/user1-4-1-4-2-20181109.mat
first CVNN path: /srv/dataset/widar/20181109/user2/user2-4-1-3-19-r5.mat


In [42]:
def parse_widar_filename(path: str):
    stem = Path(path).stem
    parts = stem.split('-')
    if len(parts) != 6:
        return {
            'raw_stem': stem,
            'parts': parts
        }

    subject, activity, scene_a, scene_b, repetition, suffix = parts

    return {
        'raw': stem,
        'subject': subject,
        'activity_id': int(activity),
        'scene_a': int(scene_a),
        'scene_b': int(scene_b),
        'repetition_id': int(repetition),
        'suffix': suffix,
        'zero_based_label_used_by_code': int(activity) - 1
    }

In [43]:
CVNN_DIR = Path("../Pytorch/CVNN")
RVNN_DIR = Path("../Pytorch/RVNN")
SEN_DIR = Path("../Pytorch/SEN")

In [44]:
print('CVNN dir exists:', CVNN_DIR.exists())
print('RVNN dir exists:', RVNN_DIR.exists())
print('SEN dir exists:', SEN_DIR.exists())

CVNN dir exists: True
RVNN dir exists: True
SEN dir exists: True


In [46]:
print('Parsed RVNN filename:')
pprint(parse_widar_filename(rvnn_paths[0]))
print('\nParsed CVNN filename:')
pprint(parse_widar_filename(cvnn_paths[0]))

Parsed RVNN filename:
{'activity_id': 4,
 'raw': 'user1-4-1-4-2-20181109',
 'repetition_id': 2,
 'scene_a': 1,
 'scene_b': 4,
 'subject': 'user1',
 'suffix': '20181109',
 'zero_based_label_used_by_code': 3}

Parsed CVNN filename:
{'activity_id': 4,
 'raw': 'user2-4-1-3-19-r5',
 'repetition_id': 19,
 'scene_a': 1,
 'scene_b': 3,
 'subject': 'user2',
 'suffix': 'r5',
 'zero_based_label_used_by_code': 3}



### How to read these shapes

- `chapter 3 / 4`: CSI tensor like `[999, 57, 3, 2]`
  - `999`: packets over time
  - `57`: subcarriers
  - `3`: antennas
  - `2`: HT-LTF slices
- `CVNN`: each sample is loaded as `cfr_array`, then interpolated to temporal length `1000`
  - model expects something that behaves like `[time=1000, feature=90]` complex
  - `90 = 3 receivers x 30 subcarriers`
- `RVNN`: each sample is loaded as `doppler_spectrum`
  - model expects something that behaves like `[6, 121, 1000]` real-valued
  - `6` usually acts like six receive links / channels
  - `121` is the Doppler-frequency axis
- `SEN`: works on complex spectrograms and tries to undo spectral leakage
  - input is not raw CSI packets
  - input is already a leaked complex spectrogram in a frequency representation

So chapter 5 is really a chapter about **AI on derived RF representations**, not about raw CSI tensors directly.


In [18]:
import concurrent.futures
import sys

def load_module(name: str, path: Path, timeout_seconds: int = 3):
    path = Path(path).resolve()
    if str(path.parent) not in sys.path:
        sys.path.insert(0, str(path.parent))
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    executor = concurrent.futures.ThreadPoolExecutor(max_workers=1)
    future = executor.submit(path.read_text, encoding='utf-8')
    try:
        source = future.result(timeout=timeout_seconds)
    except concurrent.futures.TimeoutError as exc:
        raise TimeoutError(f'Timed out while reading {path}. The file may be unavailable locally.') from exc
    finally:
        executor.shutdown(wait=False, cancel_futures=True)
    exec(compile(source, str(path), 'exec'), module.__dict__)
    return module

In [17]:
def count_parameters(model: torch.nn.Module):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

In [20]:
def trace_named_modules(model: torch.nn.Module, module_names, sample_input: torch.Tensor):
    traces = []
    hooks = []
    name_to_module = dict(model.named_modules())

    def make_hook(name):
        def hook(module, inputs, output):
            if isinstance(output, torch.Tensor):
                traces.append((name, tuple(output.shape), str(output.dtype)))
            else:
                traces.append((name, type(output).__name__, 'non-tensor'))
        return hook

    for name in module_names:
        module = name_to_module[name]
        hooks.append(module.register_forward_hook(make_hook(name)))

    with torch.no_grad():
        _ = model(sample_input)

    for hook in hooks:
        hook.remove()

    return traces

In [21]:
def display_figure(fig):
    if not MATPLOTLIB_AVAILABLE:
        return
    try:
        from IPython.display import display
        display(fig)
    except Exception:
        plt.show()


## Import the shipped PyTorch model definitions

We import the original modules from the repository rather than rewriting the networks from scratch.
That lets us answer practical questions like:

- what exact tensor shape does each model consume?
- how many parameters does it have?
- where does the complex-to-real conversion happen?
- which parts of the code still assume a remote Linux GPU training environment?


In [ ]:
module_specs = [
    ('cvnn_alexnet', '../Pytorch/CVNN/AlexNet.py'),
    ('cvnn_resnet', '../Pytorch/CVNN/Resnet_runtime.py'),
    ('cvnn_transformer', '../Pytorch/CVNN/Transformer_runtime.py'),
    ('rvnn_resnet', '../Pytorch/RVNN/ResNet.py'),
    ('sen_module', '../Pytorch/SEN/SEN_trainer.py'),
]

loaded_modules = {}

for module_name, module_path in module_specs:
    try:
        loaded_modules[module_name] = load_module(module_name, module_path)
        print(f'Loaded {module_name} from {module_path}')
    except Exception as exc:
        loaded_modules[module_name] = None
        print(f'Skipped {module_name}: {exc}')

cvnn_alexnet_mod = loaded_modules['cvnn_alexnet']
cvnn_resnet_mod = loaded_modules['cvnn_resnet']
cvnn_transformer_mod = loaded_modules['cvnn_transformer']
rvnn_resnet_mod = loaded_modules['rvnn_resnet']
sen_mod = loaded_modules['sen_module']

In [ ]:
AlexNet = cvnn_alexnet_mod.AlexNet if cvnn_alexnet_mod is not None else None
CVResNet18 = cvnn_resnet_mod.ResNet18 if cvnn_resnet_mod is not None else None
RF_Transformer = cvnn_transformer_mod.RF_Transformer if cvnn_transformer_mod is not None else None
RVResNet18 = rvnn_resnet_mod.ResNet18 if rvnn_resnet_mod is not None else None
SEN = sen_mod.SEN if sen_mod is not None else None

In [ ]:
models = {}

if AlexNet is not None:
    models['CVNN AlexNet'] = AlexNet()
if CVResNet18 is not None:
    models['CVNN ResNet18'] = CVResNet18()
if RF_Transformer is not None:
    models['CVNN RF_Transformer'] = RF_Transformer()
if RVResNet18 is not None:
    models['RVNN ResNet18'] = RVResNet18()
if SEN is not None:
    models['SEN'] = SEN(feature_len=121)

print('Available models:', ', '.join(models.keys()))

In [ ]:
for name, model in models.items():
    total, trainable = count_parameters(model)
    print(f'{name:22s} total={total:,} trainable={trainable:,}')


## **CVNN branch: complex-valued models on CFR-like inputs**

The CVNN branch keeps the complex structure of the signal.
The shipped code uses three complex-valued classifier families:

- `AlexNet`
- `ResNet18`
- `RF_Transformer`

All three finally output 6 real-valued logits for gesture classification.


In [ ]:
torch.manual_seed(0)

cvnn_input = (
    torch.randn(2, 1000, 90) + 1j * torch.randn(2, 1000, 90)
).to(torch.complex64)

cvnn_models = {}
if 'CVNN AlexNet' in models:
    cvnn_models['AlexNet'] = models['CVNN AlexNet']
if 'CVNN ResNet18' in models:
    cvnn_models['ResNet18'] = models['CVNN ResNet18']
if 'CVNN RF_Transformer' in models:
    cvnn_models['RF_Transformer'] = models['CVNN RF_Transformer']

for name, model in cvnn_models.items():
    with torch.no_grad():
        logits = model(cvnn_input)
    print(f'{name:14s} input={tuple(cvnn_input.shape)} {cvnn_input.dtype} -> output={tuple(logits.shape)} {logits.dtype}')


In [ ]:
alex_traces = trace_named_modules(
    models['CVNN AlexNet'],
    ['conv1', 'maxpool1', 'conv2', 'maxpool2', 'conv3', 'conv4', 'conv5', 'maxpool3', 'fc1', 'c2r', 'fc2', 'fc3'],
    cvnn_input,
)

print('AlexNet shape trace:')

for item in alex_traces:
    print(item)

In [ ]:
if 'CVNN ResNet18' in models:
    cvres_traces = trace_named_modules(
        models['CVNN ResNet18'],
        ['conv1', 'maxpool', 'layer1', 'layer2', 'layer3', 'layer4', 'avgpool', 'c2r', 'fc'],
        cvnn_input,
    )

    print('CVNN ResNet18 shape trace:')

    for item in cvres_traces:
        print(item)
else:
    print('CVNN ResNet18 is unavailable because ResNet18 could not be loaded from disk.')

In [ ]:
stacked = torch.stack(
    (
        torch.real(cvnn_input), 
        torch.imag(cvnn_input)
    ), 
    dim=-1
)
patched = rearrange(stacked, 'b (x s) d i -> b x (s d) i', s=8)
class_token = torch.zeros_like(patched[:, 0:1])
sequence = torch.cat((class_token, patched), dim=1)

print('Transformer patching intuition:')
print('complex CFR input shape         :', tuple(cvnn_input.shape))
print('stack real/imag shape           :', tuple(stacked.shape))
print('after ViT-like segment reshape  :', tuple(patched.shape))
print('after prepending class token    :', tuple(sequence.shape))
print('token feature width             :', patched.shape[2], 'which matches origin_dim=720 = 8 x 90')


### What the CVNN models are doing conceptually

- `AlexNet`: treats the 3 receivers as 3 complex channels and performs aggressive early convolution over time and spatial feature axes.
- `ResNet18`: keeps the same “3 receiver channels” idea but uses residual blocks to stabilize deeper feature extraction.
- `RF_Transformer`: turns the temporal axis into ViT-like patches of length 8, uses a complex Transformer encoder, then classifies from the class token.

The common pattern is:

1. keep complex numbers for as long as possible
2. extract features in the complex domain
3. convert to real only near the classifier head



## **RVNN branch: real-valued model on Doppler spectrum**

The RVNN branch operates on `doppler_spectrum`, not on complex CFR directly.
This is the more classical pipeline:

- signal preprocessing produces a Doppler-like spectrum
- the model then sees a real-valued tensor with shape close to `[6, 121, 1000]`
- classification is done with a standard real-valued ResNet


In [58]:
rvnn_input = torch.randn(2, 6, 121, 1000)

In [ ]:
with torch.no_grad():
    rvnn_logits = models['RVNN ResNet18'](rvnn_input)

In [ ]:
print('RVNN input shape :', tuple(rvnn_input.shape), rvnn_input.dtype)
print('RVNN output shape:', tuple(rvnn_logits.shape), rvnn_logits.dtype)

In [ ]:
rvnn_traces = trace_named_modules(
    models['RVNN ResNet18'],
    ['conv1', 'maxpool', 'layer1', 'layer2', 'layer3', 'layer4', 'avgpool', 'fc'],
    rvnn_input,
)

print('\nRVNN ResNet18 shape trace:')

for item in rvnn_traces:
    print(item)


### Why RVNN is easier to start with

If your mental goal is “understand the AI first”, RVNN is the gentlest entry point because:

- the input is real-valued
- the preprocessing target is interpretable as a time-frequency map
- the architecture is close to a standard image-style CNN

This makes RVNN a good baseline before diving into complex-valued layers.



## **SEN branch: enhancement network for leaked complex spectrograms**

`SEN` is not a classifier.
It is a signal-enhancement network that tries to recover a cleaner spectrum from a leaked one.

Its role in the pipeline is closer to:

- corrupted spectrogram in
- enhanced spectrogram out
- downstream sensing model later benefits from the cleaner representation


In [ ]:
blur_matrix = sen_mod.generate_blur_matrix_complex(
    wind_type="gaussian",
    wind_len=125,
    padded_len=1000,
    crop_len=121
)

In [ ]:
clean_spec, blurred_spec, leaked_spec = sen_mod.syn_one_batch_complex(
    blur_matrix_right=blur_matrix,
    feature_len=121,
    n_batch=4
)

In [ ]:
bichannel_leaked = sen_mod.complex_array_to_bichannel_float_tensor(leaked_spec)

In [ ]:
sen_input = torch.randn(2, 64, 2, 121)

In [ ]:
with torch.no_grad():
    sen_output = models['SEN'](sen_input)

In [ ]:
print('blur matrix shape          :', blur_matrix.shape)
print('synthetic clean spectrum   :', clean_spec.shape, clean_spec.dtype)
print('synthetic leaked spectrum  :', leaked_spec.shape, leaked_spec.dtype)
print('bichannel leaked tensor    :', tuple(bichannel_leaked.shape), bichannel_leaked.dtype)
print('SEN dummy input            :', tuple(sen_input.shape), sen_input.dtype)
print('SEN dummy output           :', tuple(sen_output.shape), sen_output.dtype)

In [ ]:
if MATPLOTLIB_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].imshow(np.abs(blur_matrix), aspect='auto', cmap='viridis')
    axes[0].set_title('Magnitude of the SEN blur matrix')
    axes[0].set_xlabel('Output frequency bin')
    axes[0].set_ylabel('Input frequency bin')

    axes[1].plot(np.abs(clean_spec[0]), label='ideal spectrum')
    axes[1].plot(np.abs(blurred_spec[0]), label='after window leakage')
    axes[1].plot(np.abs(leaked_spec[0]), label='after leakage + AWGN')
    axes[1].set_title('One synthetic SEN training sample')
    axes[1].set_xlabel('Frequency bin')
    axes[1].set_ylabel('Magnitude')
    axes[1].legend()

    fig.tight_layout()
    display_figure(fig)
else:
    print('Matplotlib not available, skipping SEN visualization')


### Why the SEN code needed one portability fix

The original `SEN_trainer.py` hard-coded `.cuda()` inside `m_Linear.swap_real_imag`, which breaks on a CPU-only Mac even though the math itself is device-agnostic.

For this workspace, that line was patched to use the device of the incoming tensor instead.
That means:

- same algorithm
- more portable behavior
- the notebook can run on your local machine without assuming CUDA exists



## Training loop assumptions encoded in the reference scripts

The training code is intentionally lightweight, but it makes several strong assumptions about the environment.


In [ ]:
training_notes = {
    'CVNN': {
        'dataset_key': 'cfr_array',
        'default_preprocess': 'interpolate temporal length to 1000',
        'loss': 'CrossEntropyLoss',
        'optimizer': 'SGD',
        'learning_rate': 3e-3,
        'epochs': 200,
        'batch_size': 64,
        'save_every': 5,
        'hardcoded_path_log': '/srv/csj/tutorial_submit/shuffle.txt',
        'default_device_in_script': 'cuda:0',
    },
    'RVNN': {
        'dataset_key': 'doppler_spectrum',
        'default_preprocess': 'interpolate temporal length to 1000',
        'loss': 'CrossEntropyLoss',
        'optimizer': 'SGD',
        'learning_rate': 1e-2,
        'epochs': 200,
        'batch_size': 64,
        'save_every': 5,
        'hardcoded_path_log': '/srv/csj/tutorial_dfs_submit/shuffle.txt',
        'default_device_in_script': 'cuda:0',
    },
    'SEN': {
        'task': 'spectrogram enhancement, not classification',
        'training_data': 'synthetic clean / leaked spectrum pairs',
        'loss': 'L2-like magnitude reconstruction loss',
        'optimizer': 'RMSprop',
        'learning_rate': 1e-3,
        'epochs': 10000,
        'iterations_per_epoch': 500,
        'default_device_in_script': 'CUDA-oriented training script',
    },
}

for name, notes in training_notes.items():
    print(f'===== {name} =====')
    for key, value in notes.items():
        print(f'{key:24s}: {value}')
    print()


## What this means for your learning roadmap

Recommended order inside chapter 5:

1. `RVNN ResNet18`
   - easiest mental model
   - real-valued input
   - classic CNN-style classification on DFS
2. `CVNN ResNet18` or `CVNN AlexNet`
   - same classification task
   - now you see what changes when the model keeps the complex signal structure
3. `RF_Transformer`
   - same complex input, but sequence modeling becomes the main inductive bias
4. `SEN`
   - different task family entirely
   - enhancement front-end instead of classification back-end

The key conceptual bridge from earlier chapters is:

- chapter 2: how CSI can arise
- chapter 4: why CSI must be sanitized
- chapter 3: how interpretable features are extracted
- chapter 5: how learned representations and neural networks sit on top of those signals
